# MRIxFields Etapa 2 — Gate 0 (diagnóstico)

Diagnóstico barato que decide si Gate 1 (residual objective + contrast pretraining) vale
horas de A100. **Aquí no se entrena nada.**

Lo que midió v2 en el traveller held-out 0006 (60 pares, decode full-volume):

| | nRMSE | SSIM |
|---|---|---|
| transport (SB v2) | 0.459 | 0.880 |
| identity | 0.595 | 0.876 |
| ceiling (VAE congelado) | 0.131 | 0.966 |

nRMSE se movió mucho, SSIM no se movió nada contra un techo de 0.966. Esa es la firma de un
reescalado global de intensidad, no de estructura — el mismo diagnóstico de v1, un bundle
después.

**Hipótesis que este gate prueba:** el coupling `nn` sobre un pool unpaired entrega el volumen
más cercano de OTRO sujeto en el campo destino, así que la media condicional aprendible es
aproximadamente la media poblacional de ese campo. Como el conditioning trae `log(f_t/f_s)`
explícito, "multiplicar por k" es la función más barata de ese input. Si eso es todo lo que se
aprendió, un afín cerrado por canal — sin red, sin gradientes — debería igualarlo, y lo que el
afín deje es todo el presupuesto disponible para Gate 1.

Los 5 pasos:

1. **Wrong-target sweep** — ¿el modelo responde al campo destino, o emite lo mismo siempre?
2. **Baseline afín** en forma cerrada (per-canal, per par de dominios).
3. **Gate de 4 referencias** en 0006: identity / afín / SB v2 / SB−afín, con LPIPS.
4. **SSIM post-normalización robusta** — separa brillo de estructura.
5. **Energía del residual vs piso** — el decisivo.

**0009 sigue congelado.** No se toca en ningún paso; el CLI aborta si aparece.

Umbrales pre-declarados en `configs/experiment/stage2_gate0.yaml`. Se fijan ANTES de leer
números, no después.

## 1. Entorno: GPU, Drive, rutas

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE = "/content/drive/MyDrive/MRIxFields2026"
WORK  = f"{DRIVE}/stage2_gate0"

# --- ENTRADAS: ajusta estas cuatro y nada más ------------------------------------------------
# El banco DEBE ser el full-volume. El CLI aborta si strategy_used != ["full"], así que un
# banco tiled no puede colarse por accidente.
LATENTS    = f"{DRIVE}/LatentBanks/runC_ep015_step047700_74132b9c_full_bf16"
SPLIT      = f"{DRIVE}/splits/split_v3.json"          # split PRE-resplit
VAE_CKPT   = f"{DRIVE}/runC/vae_kl_vae_best_ep15.pt"  # VAE congelado (run C v2 ep15)
SB_CKPT    = f"{DRIVE}/stage2_v2/sb_v2/ckpt/transport_sb_brownian_v2_last.pt"
# ---------------------------------------------------------------------------------------------

VAE_CONFIG = "configs/experiment/stage1_vae_v2_fgw_freebits.yaml"
SB_CONFIG  = "configs/experiment/stage2_transport_sb_v2.yaml"

Path(WORK).mkdir(parents=True, exist_ok=True)
for label, path in [("LATENTS", LATENTS), ("SPLIT", SPLIT), ("VAE_CKPT", VAE_CKPT)]:
    assert Path(path).exists(), f"STOP: {label} no existe -> {path}"
    print(f"{label:10s} OK  {path}")

HAS_SB = Path(SB_CKPT).is_file()
print(f"{'SB_CKPT':10s} {'OK' if HAS_SB else 'AUSENTE — pasos 1 y las filas SB del 3 se omiten'}  {SB_CKPT}")
print("\nWORK:", WORK)

## 2. Repositorio y dependencias

`official-evaluation` trae nibabel + scikit-image + lpips. LPIPS es obligatorio aquí: es la
tercera métrica oficial y v2 no la reportó.

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO = Path("/content/MRIxFields")
BRANCH = "experiment/stage2-gate0-diagnostic"
EXPECTED_SHA = None  # fija el sha del merge para reproducibilidad estricta

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", BRANCH,
                "https://github.com/GuillermoTafoya/MRIxFields.git", str(REPO)], check=True)
if EXPECTED_SHA:
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_SHA], check=True)

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("HEAD:", HEAD)

%cd /content/MRIxFields
!pip -q install -e ".[nifti,evaluation,official-evaluation]" scipy

for command in ("fit-affine-baseline", "gate0-sweep", "gate0-reference-gate", "gate0-residual-gate"):
    subprocess.run(["python", "-m", "fieldbridge.cli", command, "--help"],
                   check=True, stdout=subprocess.DEVNULL)
print("CLI Gate-0 verificado.")

## 3. Resplit 1-1-1 y guard del banco

`0007` train / `0006` validation / `0009` test. El resplit recomputa los fingerprints de
membresía: antes los heredaba del archivo de entrada, y como son sensibles a membresía el
archivo resultante era ilegible para `load_vae_splits` ("stale or altered"). Ese bug bloqueaba
silenciosamente que el gate usara el anchor held-out que el resplit existe para crear.

In [ ]:
import json
import subprocess
from pathlib import Path

from fieldbridge.data.resplit import resplit_file
from fieldbridge.data.vae_splits import load_vae_splits
from fieldbridge.evaluation.stage2_gate0 import assert_full_volume_bank

SPLIT_RESPLIT = f"{WORK}/split_v4_111.json"
summary = resplit_file(SPLIT, SPLIT_RESPLIT, ["P:0006"], "validation")
assert summary["counts"] == {"train": 1575, "validation": 204, "test": 205}, \
    f"STOP: conteos inesperados tras el resplit -> {summary['counts']}"

splits = load_vae_splits(SPLIT_RESPLIT)  # falla ruidosamente si el fingerprint no cuadra
travellers = lambda rs: sorted({r.subject_id for r in rs if str(r.case_id).startswith("P_")})
print("travellers  train:", travellers(splits.train),
      " validation:", travellers(splits.validation),
      " test:", travellers(splits.test))
assert travellers(splits.train) == ["0007"] and travellers(splits.validation) == ["0006"]

manifest = json.loads((Path(LATENTS) / "latent_bank_manifest.json").read_text())
print("\nbanco:", json.dumps(assert_full_volume_bank(manifest), indent=2))

## 4. Paso 2 — baseline afín en forma cerrada

Para cada par ordenado (dominio origen, dominio destino) y cada canal latente:

    a_c = sigma_t[c] / sigma_s[c]        b_c = mu_t[c] - a_c * mu_s[c]

Mínimos cuadrados necesita correspondencias y el pool retrospectivo es unpaired: no existe
`(z_s, z_t)` de la misma anatomía para regresar. Lo que el pool sí da es la marginal por canal
de cada dominio. Emparejar esas marginales **es** el mapa de transporte óptimo 1-D entre las dos
gaussianas con esos momentos, o sea el equivalente cerrado del ajuste unpaired — no una
aproximación.

Se ajustan dos tablas en la misma pasada: `all` (todos los vóxeles) y `foreground` (por encima
del percentil 50 de la norma por canal). El fondo es casi constante entre campos y arrastra los
momentos hacia "no hagas nada", así que `foreground` es la vara más exigente.

Una pasada sobre ~1575 latentes ≈ 11 GB de lectura. ~10-15 min.

In [ ]:
AFFINE_STEM = f"{WORK}/affine_baseline.json"
AFFINE_ALL  = f"{WORK}/affine_baseline_all.json"
AFFINE_FG   = f"{WORK}/affine_baseline_foreground.json"

if not Path(AFFINE_ALL).is_file():
    subprocess.run([
        "python", "-u", "-m", "fieldbridge.cli", "fit-affine-baseline",
        "--bank-dir", LATENTS,
        "--split-json", SPLIT_RESPLIT,
        "--pool-split", "train",
        "--cohort", "R",
        "--out", AFFINE_STEM,
        "--log-every", "200",
    ], check=True)
else:
    print("AFFINE YA EXISTE:", AFFINE_ALL)

from fieldbridge.models.translators.affine_baseline import AffineLatentBaseline
from fieldbridge.data.domains import Contrast, Domain

baseline = AffineLatentBaseline.load(AFFINE_FG)
print(f"\n{'par':22s} {'a (por canal)':34s} {'b (por canal)'}")
for f_s, f_t in [(0.1, 7.0), (0.1, 1.5), (3.0, 7.0), (7.0, 0.1)]:
    a, b = baseline.coefficients(Domain(f_s, Contrast.T1W), Domain(f_t, Contrast.T1W))
    print(f"T1w {f_s:>4g}T->{f_t:<4g}T   {[round(float(v), 3) for v in a]}   {[round(float(v), 3) for v in b]}")
print("\nSi a ~ 1 y b ~ 0, el afín cerrado es casi la identidad EN LATENTE, y el reescalado")
print("de intensidad que v2 hizo en imagen no es un afín por canal en latente.")

## 5. Paso 1 — wrong-target sweep

Fija un source real y pide los 5 campos destino (incluido el correcto y el propio, que es la
petición identidad). Todo en espacio latente, sin decode: prácticamente gratis.

Se reporta, por (sujeto, contraste, campo origen): distancia de cada salida al source, la matriz
completa de distancias entre las 5 salidas, y las mismas dos cantidades para los latentes
**reales** de destino — esa es la vara de cuánto debería moverse.

Veredicto por caso: `responde` si el spread de las 5 salidas alcanza al menos el
`responsiveness_min` (0.10) del spread de los 5 destinos reales. Es una vara deliberadamente
baja: pide moverse una décima parte de lo que debería.

In [ ]:
SWEEP_OUT = f"{WORK}/gate0_sweep_0006.json"

if HAS_SB:
    subprocess.run([
        "python", "-u", "-m", "fieldbridge.cli", "gate0-sweep",
        "--bank-dir", LATENTS,
        "--split-json", SPLIT_RESPLIT,
        "--subjects", "0006",
        "--transport-config", SB_CONFIG,
        "--transport-checkpoint", SB_CKPT,
        "--solver", "heun", "--n-steps", "20",
        "--out", SWEEP_OUT,
        "--device", "cuda",
    ], check=True)

    sweep = json.loads(Path(SWEEP_OUT).read_text())
    s = sweep["summary"]
    print("\n=== PASO 1 ===")
    print(f"casos                      {s['num_cases']}")
    print(f"responde al campo destino  {s['responds_fraction']:.1%}")
    print(f"monotonía en |log(ft/fs)|  {s['monotone_fraction']:.1%}  (real: {s['real_monotone_fraction']:.1%})")
    print(f"responsiveness media       {s['mean_responsiveness']:.4f}")
else:
    print("SKIP paso 1: falta SB_CKPT.")

## 6. Paso 5 — energía del residual vs piso (**el decisivo**)

Residual = `z_target_real - afin(z_source)` sobre los pares con supervisión real: 0007 (train) y
0006 (validation), 3 contrastes × 20 pares ordenados = 120 pares. **0009 excluido por config**;
el CLI aborta si aparece.

Tres preguntas, en orden creciente de importancia:

1. **¿Cuánto explica ya el afín?** Fracción de la energía del desplazamiento identidad que el
   afín cerrado se lleva.
2. **¿Lo que queda está sobre el piso?** El banco guarda la media posterior **determinista**, así
   que re-encodear el mismo volumen devuelve el latente idéntico: no hay ruido de encoding
   estocástico que medir, y el único piso real dentro del espacio latente es la cuantización
   float16. También se compara contra el **piso de anatomía**: el residual construido con el
   target del sujeto equivocado.
3. **¿Es predecible?** ¿El residual de un traveller explica el del otro? Un residual puede tener
   toda la energía que quiera y ser inaprendible si es específico del sujeto — que es el fracaso
   de v2 repetido un nivel más abajo. Esta es la que decide.

**Techo de alineación.** La fracción predecible es un coseno vóxel a vóxel entre dos cerebros
distintos, así que está topada por qué tan bien se alinean esos cerebros. El gate lo mide en vez
de suponerlo: coseno cross-sujeto, mismo dominio. En este banco da **0.345**, o sea un techo de
fracción predecible de 0.119 — no 1.0. Se reporta la fracción cruda y su porcentaje del techo.
El techo es generoso: un residual es un campo de diferencias, más alta frecuencia y menos
tolerante al desalineo que los latentes crudos con los que se mide.

Se reporta pero **no decide**: el umbral pre-declarado corre sobre la fracción cruda. Mover un
umbral pre-declarado a un estadístico elegido después de ver los números es exactamente el modo
de fallo que la pre-registración existe para evitar.

**Caveat que va pegado al número:** hay dos travellers pareados, así que la fracción predecible
es un indicador 1-vs-1, no un estadístico, y no admite intervalo de confianza.

In [ ]:
RESIDUAL_OUT = f"{WORK}/gate0_residual.json"

subprocess.run([
    "python", "-u", "-m", "fieldbridge.cli", "gate0-residual-gate",
    "--bank-dir", LATENTS,
    "--split-json", SPLIT_RESPLIT,
    "--subjects", "0006", "0007",
    "--affine-baseline", AFFINE_ALL, AFFINE_FG,
    "--out", RESIDUAL_OUT,
    "--device", "cuda",
], check=True)

residual = json.loads(Path(RESIDUAL_OUT).read_text())
print("\n=== PASO 5 ===")
print(f"pares                      {residual['num_pairs']}")
print(f"piso de cuantización f16   {residual['quantization_floor_energy']:.3e}")
alignment = next(iter(residual["baselines"].values()))["predictability"]["anatomical_alignment"]
print(f"alineación anatómica       cos {alignment['mean_cosine']:.4f} "
      f"-> techo de fracción predecible {alignment['predictable_fraction_ceiling']:.4f}\n")
header = (f"{'baseline':12s} {'E_identity':>11s} {'E_residual':>11s} {'explicado':>10s} "
          f"{'E_anatomía':>11s} {'predecible':>11s} {'% del techo':>12s}")
print(header); print("-" * len(header))
for name, block in residual["baselines"].items():
    o, p = block["overall"], block["predictability"]
    print(f"{name:12s} {o['identity_energy']:11.5f} {o['residual_energy']:11.5f} "
          f"{o['explained_fraction']:9.1%} {block['anatomy_floor']['mean_energy']:11.5f} "
          f"{p['median_predictable_fraction']:11.4f} "
          f"{p['median_predictable_fraction_of_ceiling']:11.1%}")

print("\nVEREDICTO:", residual["verdict"]["decision"])
print(residual["verdict"]["rationale"])
print("\ncaveat:", residual["verdict"]["caveat"])

## 7. Pasos 3 + 4 — gate de 4 referencias en 0006

Mismo protocolo que v2 (60 pares, decode full-volume, sampler Heun 20 pasos), con LPIPS y con
la columna extra de SSIM post-normalización robusta.

Las cuatro referencias, todas decodificadas por el mismo camino para que la diferencia aísle el
transporte:

| fila | qué es |
|---|---|
| `identity` | decode del latente source, sin transporte |
| `affine` | el afín cerrado del paso 2, sin red |
| `sb_v2` | el checkpoint SB v2 existente |
| `sb_v2_minus_affine` | `z + (SB(z) − afín(z))` |
| `ceiling` | decode del latente target real (techo del VAE congelado) |

Sobre `sb_v2_minus_affine`: la resta literal de los dos latentes de salida no sirve — sale del
manifold latente (es una imagen de diferencia de media casi cero, no un cerebro) y decodificarla
no mide nada. Lo que responde la pregunta de la fila es quitarle al **desplazamiento** de SB su
parte afín: qué aporta la red más allá del reescalado cerrado.

Coste: ~180 decodes full-volume + LPIPS. **~1-1.5 h de A100.**

In [ ]:
REFERENCE_OUT = f"{WORK}/gate0_reference_0006.json"

command = [
    "python", "-u", "-m", "fieldbridge.cli", "gate0-reference-gate",
    "--bank-dir", LATENTS,
    "--split-json", SPLIT_RESPLIT,
    "--subjects", "0006",
    "--affine-baseline", AFFINE_FG,
    "--vae-config", VAE_CONFIG,
    "--vae-checkpoint", VAE_CKPT,
    "--metrics", "ssim", "nrmse", "lpips",
    "--solver", "heun", "--n-steps", "20",
    "--out", REFERENCE_OUT,
    "--device", "cuda",
]
if HAS_SB:
    command += ["--transport-config", SB_CONFIG, "--transport-checkpoint", SB_CKPT]
subprocess.run(command, check=True)

reference = json.loads(Path(REFERENCE_OUT).read_text())
assert reference["decode"]["path_used"] == ["full"], \
    f"STOP: decode cayó a {reference['decode']['path_used']} — los números cargan la aproximación tiled."
print("\ndecode:", reference["decode"])

## 8. Reporte — las 4 tablas + el veredicto

In [ ]:
def table(block, method_names, title):
    lines = [f"**{title}** ({block['num_pairs']} pares)", "",
             "| referencia | nRMSE | SSIM | SSIM robusto | LPIPS |", "|---|---|---|---|---|"]
    for name in method_names:
        m = block["methods"][name]
        cell = lambda key: f"{m[key]:.4f}" if key in m else "—"
        lines.append(f"| {name} | {cell('nrmse')} | {cell('ssim')} | {cell('ssim_robust')} | {cell('lpips')} |")
    return "\n".join(lines) + "\n"

names = reference["method_names"]
strata = reference["strata"]
report = [
    "# Gate 0 — resultados\n",
    f"Sujeto held-out: {reference['subjects']}. Decode: {reference['decode']['path_used']}.\n",
    table(reference["overall"], names, "Agregado"),
    table(strata["catastrophic_identity"], names,
          f"Estrato catastrófico ({strata['catastrophic_identity']['definition']}, "
          f"{strata['catastrophic_identity']['fraction_of_pairs']:.1%} de los pares)"),
    table(strata["ordinary"], names, "Estrato ordinario"),
]
for contrast, block in reference["by_contrast"].items():
    report.append(table(block, names, f"Contraste {contrast}"))

report.append("## Paso 5 — veredicto\n")
report.append(f"- decisión: **{residual['verdict']['decision']}**")
report.append(f"- {residual['verdict']['rationale']}")
report.append(f"- caveat: {residual['verdict']['caveat']}\n")

text = "\n".join(report)
Path(f"{WORK}/GATE0_REPORT.md").write_text(text, encoding="utf-8")
print(text)